#**Setting up vector store for Import Policies**

In [6]:
import getpass
import os

os.environ["GOOGLE_API_KEY"] = (__import__('sys').path.insert(0, '..') or __import__('config').GOOGLE_API_KEY)

##For Import Policies

In [ ]:
import os
import json
import re
import pathlib
from google import genai
from google.genai import types

def generate_chapter_context_json(pdf_path: str) -> dict:
    filepath = pathlib.Path(pdf_path)
    if not filepath.is_file():
        raise FileNotFoundError(f"The file was not found at: {pdf_path}")

    print(f"\n--- Processing PDF: {filepath.name} ---")

    client = genai.Client()
    pdf_bytes = filepath.read_bytes()

    prompt = """
    You are a data extraction expert specializing in government trade policy documents.
    Your task is to analyze the provided PDF of an ITC(HS) Import Policy chapter
    and extract all textual notes into a structured JSON format.

    Follow these instructions precisely:
    1.  **Identify the Chapter:** Find the chapter number and the chapter title.
    2.  **Extract Section Notes:** Find the first "Main Notes" section at the top of the document (these belong to the entire Section, e.g., "Section I"). Extract all notes under it.
    3.  **Extract Chapter Notes:** Find the second "Main Notes" section that is specific to the chapter itself. Extract all notes under it.
    4.  **Extract Policy Conditions:** Find the "Policy Conditions" section and extract all conditions. It is crucial to preserve the original serial number for each condition (e.g., '[05]', '[1]', '[2]').
    5.  **IGNORE:** Do NOT extract any data from the large table titled "Product Description and Import Policy" that contains HS Codes.
    6.  **Output Format:** Present the extracted data as a single, clean JSON object. Do not include any text or markdown formatting before or after the JSON. The JSON should have the following keys:
        - "chapter_no": string
        - "chapter_title": string
        - "section_main_notes": an array of strings.
        - "chapter_specific_notes": an array of strings.
        - "policy_conditions": an array of strings, where each string starts with its serial number.
    """

    print("Generating structured content from PDF...")
    response = client.models.generate_content(
        model=(__import__('sys').path.insert(0, '..') or __import__('config').LLM_MODEL),
        contents=[
            types.Part.from_bytes(data=pdf_bytes, mime_type='application/pdf'),
            prompt
        ]
    )

    try:
        json_match = re.search(r'```json\s*(\{.*?\})\s*```', response.text, re.DOTALL)
        if json_match:
            json_string = json_match.group(1)
        else:
            json_string = response.text

        extracted_data = json.loads(json_string)
        print("✅ JSON parsed successfully.")

    except (json.JSONDecodeError, AttributeError) as e:
        print(f"❌ Error parsing JSON for file {filepath.name}: {e}")
        print("Raw Response:\n", response.text)
        return {}

    context_parts = []
    context_parts.append("==START SECTION MAIN NOTES==")
    context_parts.extend(extracted_data.get("section_main_notes", []))
    context_parts.append("==END SECTION MAIN NOTES==\n")

    context_parts.append("==START CHAPTER NOTES==")
    context_parts.extend(extracted_data.get("chapter_specific_notes", []))
    context_parts.append("==END CHAPTER NOTES==\n")

    context_parts.append("==START POLICY CONDITIONS==")
    context_parts.extend(extracted_data.get("policy_conditions", []))
    context_parts.append("==END POLICY CONDITIONS==")

    full_context_text = "\n".join(context_parts)

    final_db_object = {
        "chapter_no": int(extracted_data.get("chapter_no", 0)),
        "chapter_title": extracted_data.get("chapter_title", "Unknown"),
        "full_context_text": full_context_text,
        "source_file": filepath.name
    }

    return final_db_object

def process_all_pdfs_in_folder(pdf_folder_path: str):
    pdf_folder = pathlib.Path(pdf_folder_path)
    if not pdf_folder.is_dir():
        raise NotADirectoryError(f"{pdf_folder_path} is not a valid directory.")

    all_json_data = []
    for pdf_file in sorted(pdf_folder.glob("*.pdf")):
        try:
            chapter_json = generate_chapter_context_json(str(pdf_file))
            if chapter_json:
                all_json_data.append(chapter_json)
        except Exception as e:
            print(f"⚠️ Skipping {pdf_file.name} due to error: {e}")

    return all_json_data

# --- Main Entry ---
if __name__ == '__main__':
    try:
        if 'GOOGLE_API_KEY' not in os.environ:
            os.environ["GOOGLE_API_KEY"] = (__import__('sys').path.insert(0, '..') or __import__('config').GOOGLE_API_KEY)
        genai.configure(api_key=os.environ['GOOGLE_API_KEY'])
    except Exception as e:
        print("Please configure your Google API Key.")
        exit()

    folder_path = './chapter_pdfs'  # <-- Update this to your folder path
    all_data = process_all_pdfs_in_folder(folder_path)

    if all_data:
        output_file = "all_chapters_context.json"
        with open(output_file, 'w') as f:
            json.dump(all_data, f, indent=2)
        print(f"\n✅ All chapter contexts saved to: {output_file}")
    else:
        print("❌ No valid data was extracted.")


##For Export Policies

In [ ]:
import os
import json
import re
import pathlib
from google import genai
from google.genai import types

def generate_chapter_context_json(pdf_path: str) -> dict:
    filepath = pathlib.Path(pdf_path)
    if not filepath.is_file():
        raise FileNotFoundError(f"The file was not found at: {pdf_path}")

    print(f"\n--- Processing PDF: {filepath.name} ---")

    client = genai.Client()
    pdf_bytes = filepath.read_bytes()

    prompt = """
    You are a data extraction expert specializing in government trade policy documents.
    Your task is to analyze the provided PDF of an ITC(HS) Export Policy chapter
    and extract all textual notes into a structured JSON format.

    Follow these instructions precisely:
    1.  **Identify the Chapter:** Find the chapter number and the chapter title.
    2.  **Extract Section Notes:** Find the first "Main Notes" section at the top of the document (these belong to the entire Section, e.g., "Section I"). Extract all notes under it.
    3.  **Extract Chapter Notes:** Find the second "Main Notes" section that is specific to the chapter itself. Extract all notes under it.
    4.  **Extract Policy Conditions:** Find the "Policy Conditions" section and extract all conditions. It is crucial to preserve the original serial number for each condition (e.g., '[05]', '[1]', '[2]').
    5.  **IGNORE:** Do NOT extract any data from the large table titled "Product Description and Export Policy" that contains HS Codes.
    6.  **Output Format:** Present the extracted data as a single, clean JSON object. Do not include any text or markdown formatting before or after the JSON. The JSON should have the following keys:
        - "chapter_no": string
        - "chapter_title": string
        - "section_main_notes": an array of strings.
        - "chapter_specific_notes": an array of strings.
        - "policy_conditions": an array of strings, where each string starts with its serial number.
    """

    print("Generating structured content from PDF...")
    response = client.models.generate_content(
        model=(__import__('sys').path.insert(0, '..') or __import__('config').LLM_MODEL),
        contents=[
            types.Part.from_bytes(data=pdf_bytes, mime_type='application/pdf'),
            prompt
        ]
    )

    try:
        json_match = re.search(r'```json\s*(\{.*?\})\s*```', response.text, re.DOTALL)
        if json_match:
            json_string = json_match.group(1)
        else:
            json_string = response.text

        extracted_data = json.loads(json_string)
        print("✅ JSON parsed successfully.")

    except (json.JSONDecodeError, AttributeError) as e:
        print(f"❌ Error parsing JSON for file {filepath.name}: {e}")
        print("Raw Response:\n", response.text)
        return {}

    context_parts = []
    context_parts.append("==START SECTION MAIN NOTES==")
    context_parts.extend(extracted_data.get("section_main_notes", []))
    context_parts.append("==END SECTION MAIN NOTES==\n")

    context_parts.append("==START CHAPTER NOTES==")
    context_parts.extend(extracted_data.get("chapter_specific_notes", []))
    context_parts.append("==END CHAPTER NOTES==\n")

    context_parts.append("==START EXPORT POLICY CONDITIONS==")
    context_parts.extend(extracted_data.get("policy_conditions", []))
    context_parts.append("==END EXPORT POLICY CONDITIONS==")

    full_context_text = "\n".join(context_parts)

    final_db_object = {
        "chapter_no": int(extracted_data.get("chapter_no", 0)),
        "chapter_title": extracted_data.get("chapter_title", "Unknown"),
        "full_context_text": full_context_text,
        "source_file": filepath.name
    }

    return final_db_object

def process_all_pdfs_in_folder(pdf_folder_path: str):
    pdf_folder = pathlib.Path(pdf_folder_path)
    if not pdf_folder.is_dir():
        raise NotADirectoryError(f"{pdf_folder_path} is not a valid directory.")

    all_json_data = []
    for pdf_file in sorted(pdf_folder.glob("*.pdf")):
        try:
            chapter_json = generate_chapter_context_json(str(pdf_file))
            if chapter_json:
                all_json_data.append(chapter_json)
        except Exception as e:
            print(f"⚠️ Skipping {pdf_file.name} due to error: {e}")

    return all_json_data

# --- Main Entry ---
if __name__ == '__main__':
    try:
        if 'GOOGLE_API_KEY' not in os.environ:
            os.environ["GOOGLE_API_KEY"] = (__import__('sys').path.insert(0, '..') or __import__('config').GOOGLE_API_KEY)
        genai.configure(api_key=os.environ['GOOGLE_API_KEY'])
    except Exception as e:
        print("Please configure your Google API Key.")
        exit()

    folder_path = './chapter_pdfs'  # <-- Update this to your folder path
    all_data = process_all_pdfs_in_folder(folder_path)

    if all_data:
        output_file = "all_chapters_context.json"
        with open(output_file, 'w') as f:
            json.dump(all_data, f, indent=2)
        print(f"\n✅ All chapter contexts saved to: {output_file}")
    else:
        print("❌ No valid data was extracted.")


Please configure your Google API Key.

--- Processing PDF: 067e1dd4-eb21-41c1-8db1-244690f6c507.pdf ---
Generating structured content from PDF...
✅ JSON parsed successfully.

--- Processing PDF: 08431ead-993f-4f33-9e85-b238e8a4c613.pdf ---
Generating structured content from PDF...
✅ JSON parsed successfully.

--- Processing PDF: 0bc8702c-ac4c-4757-bf94-89bb7a2c2961.pdf ---
Generating structured content from PDF...
✅ JSON parsed successfully.

--- Processing PDF: 10a30462-0a20-4d16-9632-cf2c597f1800.pdf ---
Generating structured content from PDF...
✅ JSON parsed successfully.

--- Processing PDF: 121a310b-f49c-4f0c-bbec-f02baaa3b4ed.pdf ---
Generating structured content from PDF...
✅ JSON parsed successfully.

--- Processing PDF: 129798ba-ef77-4ca4-8ad9-45849e92b062.pdf ---
Generating structured content from PDF...
✅ JSON parsed successfully.

--- Processing PDF: 19e070ac-d247-4cb1-804d-d5c12a0cee40.pdf ---
Generating structured content from PDF...
✅ JSON parsed successfully.

--- Proce

In [ ]:
import json

def sort_by_chapter_number(json_list: list) -> list:
    """
    Sorts a list of chapter context JSON objects by the 'chapter_no' key in ascending order.
    """
    return sorted(json_list, key=lambda x: int(x.get("chapter_no", 0)))

# Load the unsorted JSON
with open("all_chapters_context.json", "r") as f:
    data = json.load(f)

# Sort by chapter_no
sorted_data = sort_by_chapter_number(data)

# Save the sorted JSON
with open("all_chapters_context_sorted.json", "w") as f:
    json.dump(sorted_data, f, indent=2)

print("✅ Sorted by chapter_no and saved to 'all_chapters_context_sorted.json'")


✅ Sorted by chapter_no and saved to 'all_chapters_context_sorted.json'


In [ ]:
!pip install langchain langchain-postgres langchain-google-genai sqlalchemy psycopg2-binary pgvector


In [ ]:
import os
import json
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_postgres.vectorstores import PGVector

# === CONFIGURATION ===
POSTGRES_CONNECTION_STRING = (__import__('sys').path.insert(0, '..') or __import__('config').DB_CONNECTION_STRING)  # 🔁 Replace with yours
os.environ["GOOGLE_API_KEY"] = (__import__('sys').path.insert(0, '..') or __import__('config').GOOGLE_API_KEY)
JSON_FILE = "all_chapters_context_sorted.json"
COLLECTION_NAME = "export_policy_chapter_context_collection"  # used as namespace

# === INITIALIZE GOOGLE GEMINI EMBEDDINGS ===
embedding_model = GoogleGenerativeAIEmbeddings(
    model="models/embedding-001",
    google_api_key=(__import__('sys').path.insert(0, '..') or __import__('config').GOOGLE_API_KEY)
)

# === LOAD CHAPTER CONTEXT DATA ===
with open(JSON_FILE, "r") as f:
    chapters = json.load(f)

documents = []
metadatas = []

for chapter in chapters:
    documents.append(chapter["full_context_text"])
    metadatas.append({
        "chapter_no": chapter["chapter_no"],
        "chapter_title": chapter["chapter_title"],
        "source_file": chapter["source_file"]
    })

# === CONNECT AND INSERT EMBEDDINGS INTO PGVECTOR ===
print("🔄 Inserting data into PostgreSQL with Gemini embeddings...")

vectorstore = PGVector.from_texts(
    texts=documents,
    embedding=embedding_model,
    metadatas=metadatas,
    collection_name=COLLECTION_NAME,
    connection=POSTGRES_CONNECTION_STRING,  # ← Key difference in the new API
)

print(f"✅ Inserted {len(documents)} documents into pgvector collection '{COLLECTION_NAME}'")


🔄 Inserting data into PostgreSQL with Gemini embeddings...
✅ Inserted 96 documents into pgvector collection 'export_policy_chapter_context_collection'


#Now We'll extract the master table.

In [1]:
import os
import json
import re
import pathlib
from google import genai
from google.genai import types

def extract_hscode_table_json(pdf_path: str) -> list:
    filepath = pathlib.Path(pdf_path)
    if not filepath.is_file():
        raise FileNotFoundError(f"The file was not found at: {pdf_path}")

    print(f"\n--- Processing PDF: {filepath.name} ---")

    client = genai.Client()
    pdf_bytes = filepath.read_bytes()

    prompt = """
    Follow these instructions precisely:

    1. Extract Structured Data: From the large tabular section titled "Product Description and Import Policy", extract each row as a structured entry.

    2. Required Fields per Row:
       - "RoDTEP Entry" : string
       - "Tarrif Lines": string
       - "Description of Goods (As per CTH )": string
       - "uqc": string
       - "Rate as % age of FOB": string (must include '%' symbol)
       - "Cap (Rs. Per UQC)": string (leave empty if not available)

    3. IGNORE: Do NOT extract any headings, summary text, chapter notes, section notes, or unrelated metadata. Focus only on the structured table rows.

    4. Data Formatting Rules:
       - Convert numeric rate values like 1.4 into string format with a percentage symbol, such as "1.4%".
       - If cap_per_uqc is not available, set it as an empty string "".
       - Ensure all field values are returned as strings.
       - Do not return any null, None, or non-JSON types.

    5. Output Format: Present the extracted data as a single, clean JSON object. Do not include any text or markdown formatting before or after the JSON. The JSON should have the following keys:
       - "entries": an array of objects, each containing:
        - "RoDTEP Entry" : string
        - "Tarrif Lines": string
        - "Description of Goods (As per CTH )": string
        - "uqc": string
        - "Rate as % age of FOB": string (must include '%' symbol)
        - "Cap (Rs. Per UQC)": string (leave empty if not available)
    """

    print("Generating structured HS Code content from PDF...")
    response = client.models.generate_content(
        model=(__import__('sys').path.insert(0, '..') or __import__('config').LLM_MODEL),
        contents=[
            types.Part.from_bytes(data=pdf_bytes, mime_type='application/pdf'),
            prompt
        ]
    )

    try:
        # Extract valid JSON portion
        json_match = re.search(r'```json\s*({.*?})\s*```', response.text, re.DOTALL)
        json_string = json_match.group(1) if json_match else response.text

        parsed = json.loads(json_string)
        entries = parsed.get("entries", [])

        # Normalize and enrich entries
        for entry in entries:
            entry["Rate as % age of FOB"] = f"{entry['Rate as % age of FOB'].strip('%')}%" if entry.get("Rate as % age of FOB") else ""
            entry["Cap (Rs. Per UQC)"] = entry.get("Cap (Rs. Per UQC)", "") or ""
            entry["uqc"] = str(entry.get("uqc", "")).strip()

            # Add enrichment placeholders
            for field in [
                "import_policy", "notification_no", "notification_date",
                "policy_condition", "item_notes", "restriction_type",
                "ftp_2023_category", "chapter_no"
            ]:
                entry[field] = ""

        print("✅ JSON parsed successfully.")
        return entries

    except Exception as e:
        print(f"❌ Error parsing JSON for file {filepath.name}: {e}")
        print("Raw Gemini Response:\n", response.text[:3000])  # Shorten for debug
        return []

def process_all_hscode_pdfs(pdf_folder_path: str):
    pdf_folder = pathlib.Path(pdf_folder_path)
    if not pdf_folder.is_dir():
        raise NotADirectoryError(f"{pdf_folder_path} is not a valid directory.")

    all_entries = []
    for pdf_file in sorted(pdf_folder.glob("*.pdf")):
        try:
            page_entries = extract_hscode_table_json(str(pdf_file))
            all_entries.extend(page_entries)
        except Exception as e:
            print(f"⚠️ Skipping {pdf_file.name} due to error: {e}")

    return all_entries

# --- Main Entry ---
if __name__ == '__main__':
    try:
        if 'GOOGLE_API_KEY' not in os.environ:
            os.environ['GOOGLE_API_KEY'] = (__import__('sys').path.insert(0, '..') or __import__('config').GOOGLE_API_KEY)
        genai.configure(api_key=os.environ['GOOGLE_API_KEY'])
    except Exception as e:
        print("Please configure your Google API Key.")
        exit()

    folder_path = '/content/HS_codes'  # 📁 Folder containing all HS Code PDFs
    all_data = process_all_hscode_pdfs(folder_path)

    if all_data:
        output_file_json = "master_hscode_10.json"
        output_file_jsonl = "master_hscode1_10.jsonl"

        with open(output_file_json, 'w', encoding='utf-8') as f:
            json.dump(all_data, f, indent=2)

        with open(output_file_jsonl, 'w', encoding='utf-8') as f:
            for obj in all_data:
                f.write(json.dumps(obj) + "\n")

        print(f"\n✅ All HS Code entries saved to: {output_file_json} and {output_file_jsonl}")
    else:
        print("❌ No valid HS Code data was extracted.")


Please configure your Google API Key.

--- Processing PDF: Appendix+4RE+wef+10+October+2024 (1)_page_116.pdf ---
Generating structured HS Code content from PDF...
✅ JSON parsed successfully.

--- Processing PDF: Appendix+4RE+wef+10+October+2024 (1)_page_117.pdf ---
Generating structured HS Code content from PDF...
✅ JSON parsed successfully.

--- Processing PDF: Appendix+4RE+wef+10+October+2024 (1)_page_118.pdf ---
Generating structured HS Code content from PDF...
✅ JSON parsed successfully.

--- Processing PDF: Appendix+4RE+wef+10+October+2024 (1)_page_119.pdf ---
Generating structured HS Code content from PDF...
✅ JSON parsed successfully.

--- Processing PDF: Appendix+4RE+wef+10+October+2024 (1)_page_120.pdf ---
Generating structured HS Code content from PDF...
✅ JSON parsed successfully.

--- Processing PDF: Appendix+4RE+wef+10+October+2024 (1)_page_121.pdf ---
Generating structured HS Code content from PDF...
✅ JSON parsed successfully.

--- Processing PDF: Appendix+4RE+wef+10+Oct

In [1]:
import json

# ✅ Only these fields will be renamed
FIELD_MAPPING = {
    "hs_code": "Tariff Lines",
    "description": "Description of Goods (As per CTH )",
    "uqc": "UQC",
    "rate_percent": "Rate as % age of FOB (#)",
    "cap_per_uqc": "Cap (Rs. Per UQC)",
    "chapter_number" : "chapter_no"
}

def transform_entry(entry):
    transformed = {}
    for key, value in entry.items():
        new_key = FIELD_MAPPING.get(key, key)  # Rename if in mapping, else keep as is
        transformed[new_key] = value
    return transformed

# 📂 Input and Output files
input_file = "master_hscode.json"
output_file = "transformed_hscode_with_extras.json"

# 🔄 Load and transform
with open(input_file, "r", encoding="utf-8") as f:
    data = json.load(f)

transformed_data = [transform_entry(entry) for entry in data]

# 💾 Save
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(transformed_data, f, indent=2)

print(f"✅ JSON with renamed + extra fields saved to: {output_file}")


FileNotFoundError: [Errno 2] No such file or directory: 'master_hscode.json'

In [2]:
import json

# 📂 File paths
input_file = "/content/master_hscode_10.json"
output_file = "sorted_hscode.json"

# 📥 Load data
with open(input_file, "r", encoding="utf-8") as f:
    data = json.load(f)

# 🔢 Sort by "Tariff Lines" (numerically, not as strings)
sorted_data = sorted(data, key=lambda x: int(x["Tarrif Lines"]))

# 💾 Save sorted result
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(sorted_data, f, indent=2)

print(f"✅ Sorted JSON saved to: {output_file}")


✅ Sorted JSON saved to: sorted_hscode.json


###Json From Chapters for Import

In [ ]:
import os
import json
import re
import pathlib
from google import genai
from google.genai import types

# --- Gemini Prompt to extract from the HS table ---
EXTRACTION_PROMPT = """
You are an expert in extracting structured data from trade policy PDFs.

Your task is to extract all rows from the table titled **"Product Description and Import Policy"** from the given chapter document.

For each row, extract the following fields:
- "HS code" (Tariff item number)
- "Import Policy"
- "Policy Condition"
- "Notification No."
- "Notification Date"
- "chapter_no"

Rules:
1. Only extract from the table. Ignore notes or summaries outside the table.
2. All fields must be returned as strings.
3. If any value is missing in the row, return it as an empty string "".
4. Add a sixth field manually: `"chapter_no"` as a string (e.g., "03", "04", etc.) based on the input file.
5. Output must be a single JSON object:
   {
     "entries": [ ... ]
   }
6. Do NOT include any markdown or explanation. Only return pure JSON.
"""

# --- Extract entries from a single PDF ---
def extract_chapter_policy_table(pdf_path: str) -> list:
    filepath = pathlib.Path(pdf_path)
    if not filepath.is_file():
        raise FileNotFoundError(f"❌ File not found: {pdf_path}")

    print(f"\n--- Processing: {filepath.name} ---")

    client = genai.Client()
    pdf_bytes = filepath.read_bytes()

    # Determine chapter number from filename (e.g., "chapter_3.pdf" → "03")
    chapter_number = re.search(r'chapter[_\-]?(\d+)', filepath.stem, re.IGNORECASE)
    chapter_no = chapter_number.group(1).zfill(2) if chapter_number else "00"

    # Prepare Gemini call
    response = client.models.generate_content(
        model=(__import__('sys').path.insert(0, '..') or __import__('config').LLM_MODEL),
        contents=[
            types.Part.from_bytes(data=pdf_bytes, mime_type='application/pdf'),
            EXTRACTION_PROMPT
        ]
    )

    # Try extracting JSON
    try:
        json_match = re.search(r'```json\s*({.*?})\s*```', response.text, re.DOTALL)
        json_string = json_match.group(1) if json_match else response.text
        parsed = json.loads(json_string)
        entries = parsed.get("entries", [])

        # Enforce schema and inject chapter_no
        for entry in entries:
            entry["HS code"] = str(entry.get("HS code", "")).strip()
            entry["Import Policy"] = str(entry.get("Import Policy", "")).strip()
            entry["Policy Condition"] = str(entry.get("Policy Condition", "")).strip()
            entry["Notification No."] = str(entry.get("Notification No.", "")).strip()
            entry["Notification Date"] = str(entry.get("Notification Date", "")).strip()
            entry["chapter_no"] = chapter_no  # ✅ Add chapter number

        print(f"✅ Extracted {len(entries)} entries.")
        return entries

    except Exception as e:
        print(f"❌ Failed to extract JSON: {e}")
        print("Response Text:\n", response.text[:2000])
        return []

# --- Process all PDFs in folder ---
def process_chapter_folder(folder_path: str):
    folder = pathlib.Path(folder_path)
    if not folder.is_dir():
        raise NotADirectoryError(f"{folder_path} is not a valid directory.")

    all_entries = []
    for pdf_file in sorted(folder.glob("*.pdf")):
        try:
            entries = extract_chapter_policy_table(str(pdf_file))
            all_entries.extend(entries)
        except Exception as e:
            print(f"⚠️ Skipping {pdf_file.name} due to error: {e}")

    return all_entries

# --- Main Execution ---
if __name__ == '__main__':
    if 'GOOGLE_API_KEY' not in os.environ:
        os.environ['GOOGLE_API_KEY'] = (__import__('sys').path.insert(0, '..') or __import__('config').GOOGLE_API_KEY)  # Replace this!
    # genai.configure(api_key=os.environ['GOOGLE_API_KEY'])

    # 📁 Update this folder path to your chapter PDFs
    chapters_folder = "/content/5_Chapters"
    output_file = "import_policy_combined.json"

    combined_entries = process_chapter_folder(chapters_folder)

    if combined_entries:
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(combined_entries, f, indent=2)
        print(f"\n✅ All chapter entries saved to: {output_file}")
    else:
        print("❌ No data extracted.")



--- Processing: 392d34f2-ec7b-4061-88d9-b640afafceb2.pdf ---
✅ Extracted 31 entries.

--- Processing: 5aed4eba-ea0b-46f6-8d0a-83146c8e4834.pdf ---
✅ Extracted 71 entries.

--- Processing: 7747a702-b18b-48b9-b102-d05857fe8728.pdf ---
✅ Extracted 341 entries.

--- Processing: 94fdd40a-a5ed-475d-9acd-d7c69f62f7db.pdf ---
✅ Extracted 104 entries.

--- Processing: 97b85202-f85f-4961-9eab-98f3d1a98fe5.pdf ---
✅ Extracted 114 entries.

--- Processing: 998ebb2b-03c8-4fa4-8075-6688dae936a6.pdf ---
✅ Extracted 92 entries.

✅ All chapter entries saved to: import_policy_combined.json


###Json for chapter for Export

In [9]:
import os
import json
import re
import pathlib
from google import genai
from google.genai import types

# --- Gemini Prompt to extract from the HS table ---
EXTRACTION_PROMPT = """
You are an expert in extracting structured data from trade policy PDFs.

Your task is to extract all rows from the table titled **"Product Description and Export Policy"** from the given chapter document.

For each row, extract the following fields:
- "HS code" (Tariff item number)
- "Export Policy"
- "Policy Condition"
- "Notification No."
- "Notification Date"
- "chapter_no"

Rules:
1. Only extract from the table. Ignore notes or summaries outside the table.
2. All fields must be returned as strings.
3. If any value is missing in the row, return it as an empty string "".
4. Add a sixth field manually: `"chapter_no"` as a string (e.g., "03", "04", etc.) based on the input file.
5. Output must be a single JSON object:
   {
     "entries": [ ... ]
   }
6. Do NOT include any markdown or explanation. Only return pure JSON.
"""

# --- Extract entries from a single PDF ---
def extract_chapter_policy_table(pdf_path: str) -> list:
    filepath = pathlib.Path(pdf_path)
    if not filepath.is_file():
        raise FileNotFoundError(f"❌ File not found: {pdf_path}")

    print(f"\n--- Processing: {filepath.name} ---")

    client = genai.Client()
    pdf_bytes = filepath.read_bytes()

    # Determine chapter number from filename (e.g., "chapter_3.pdf" → "03")
    chapter_number = re.search(r'chapter[_\-]?(\d+)', filepath.stem, re.IGNORECASE)
    chapter_no = chapter_number.group(1).zfill(2) if chapter_number else "00"

    # Prepare Gemini call
    response = client.models.generate_content(
        model=(__import__('sys').path.insert(0, '..') or __import__('config').LLM_MODEL),
        contents=[
            types.Part.from_bytes(data=pdf_bytes, mime_type='application/pdf'),
            EXTRACTION_PROMPT
        ]
    )

    # Try extracting JSON
    try:
        json_match = re.search(r'```json\s*({.*?})\s*```', response.text, re.DOTALL)
        json_string = json_match.group(1) if json_match else response.text
        parsed = json.loads(json_string)
        entries = parsed.get("entries", [])

        # Enforce schema and inject chapter_no
        for entry in entries:
            entry["HS code"] = str(entry.get("HS code", "")).strip()
            entry["Export Policy"] = str(entry.get("Export Policy", "")).strip()
            entry["Policy Condition"] = str(entry.get("Policy Condition", "")).strip()
            entry["Notification No."] = str(entry.get("Notification No.", "")).strip()
            entry["Notification Date"] = str(entry.get("Notification Date", "")).strip()
            entry["chapter_no"] = chapter_no  # ✅ Add chapter number

        print(f"✅ Extracted {len(entries)} entries.")
        return entries

    except Exception as e:
        print(f"❌ Failed to extract JSON: {e}")
        print("Response Text:\n", response.text[:2000])
        return []

# --- Process all PDFs in folder ---
def process_chapter_folder(folder_path: str):
    folder = pathlib.Path(folder_path)
    if not folder.is_dir():
        raise NotADirectoryError(f"{folder_path} is not a valid directory.")

    all_entries = []
    for pdf_file in sorted(folder.glob("*.pdf")):
        try:
            entries = extract_chapter_policy_table(str(pdf_file))
            all_entries.extend(entries)
        except Exception as e:
            print(f"⚠️ Skipping {pdf_file.name} due to error: {e}")

    return all_entries

# --- Main Execution ---
if __name__ == '__main__':
    if 'GOOGLE_API_KEY' not in os.environ:
        os.environ['GOOGLE_API_KEY'] = (__import__('sys').path.insert(0, '..') or __import__('config').GOOGLE_API_KEY)  # Replace this!
    # genai.configure(api_key=os.environ['GOOGLE_API_KEY'])

    # 📁 Update this folder path to your chapter PDFs
    chapters_folder = "/content/export_chapters"  # Update as needed
    output_file = "export_policy_combined.json"

    combined_entries = process_chapter_folder(chapters_folder)

    if combined_entries:
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(combined_entries, f, indent=2)
        print(f"\n✅ All chapter entries saved to: {output_file}")
    else:
        print("❌ No data extracted.")



--- Processing: b6ad61a7-1fdd-48d1-a188-0bc106cb37f6.pdf ---
❌ Failed to extract JSON: expected string or bytes-like object, got 'NoneType'
⚠️ Skipping b6ad61a7-1fdd-48d1-a188-0bc106cb37f6.pdf due to error: 'NoneType' object is not subscriptable
❌ No data extracted.


#For **'hs_code_rodtep_&_import_policy'** table

In [ ]:
import json

def merge_common_trade_jsons(rodtep_json, import_policy_json):
    # Convert Import Policy to a dict for quick lookup
    import_policy_map = {item.get("HS code"): item for item in import_policy_json}

    merged_data = []

    # Loop through RoDTEP items and check for common keys
    for item in rodtep_json:
        key = item.get("Tarrif Lines")
        if key in import_policy_map:
            ip_item = import_policy_map[key]
            merged_data.append({
                "Tariff Lines / HS Code": key,
                "Description of Goods (As per CTH )": item.get("Description of Goods (As per CTH )", ""),
                "uqc": item.get("uqc", ""),
                "Rate as % age of FOB": item.get("Rate as % age of FOB", ""),
                "Cap (Rs. Per UQC)": item.get("Cap (Rs. Per UQC)", ""),
                "Import Policy": ip_item.get("Import Policy", ""),
                "Policy Condition": ip_item.get("Policy Condition", ""),
                "Notification No.": ip_item.get("Notification No.", ""),
                "Notification Date": ip_item.get("Notification Date", ""),
                "chapter_no": ip_item.get("chapter_no", "")
            })

    return merged_data

if __name__ == "__main__":
    # Load RoDTEP and Import Policy JSONs
    with open("/content/master_hscode_10.json", "r") as f:
        rodtep_json = json.load(f)

    with open("/content/import_policy_combined.json", "r") as f:
        import_policy_json = json.load(f)

    # Merge only common codes
    merged_result = merge_common_trade_jsons(rodtep_json, import_policy_json)

    # Write to output file
    with open("merged_output.json", "w") as f:
        json.dump(merged_result, f, indent=2)

    print("✅ Merged (common only) output written to 'merged_output.json'")


✅ Merged (common only) output written to 'merged_output.json'


#For Export **'hs_code_rodtep_&_export_policy'** table

In [11]:
import json

def merge_common_trade_jsons(rodtep_json, export_policy_json):
    # Convert Export Policy to a dict for quick lookup
    export_policy_map = {item.get("HS code"): item for item in export_policy_json}

    merged_data = []

    # Loop through RoDTEP items and check for common keys
    for item in rodtep_json:
        key = item.get("Tarrif Lines")
        if key in export_policy_map:
            ep_item = export_policy_map[key]
            merged_data.append({
                "Tariff Lines / HS Code": key,
                "Description of Goods (As per CTH )": item.get("Description of Goods (As per CTH )", ""),
                "uqc": item.get("uqc", ""),
                "Rate as % age of FOB": item.get("Rate as % age of FOB", ""),
                "Cap (Rs. Per UQC)": item.get("Cap (Rs. Per UQC)", ""),
                "Export Policy": ep_item.get("Export Policy", ""),
                "Policy Condition": ep_item.get("Policy Condition", ""),
                "Notification No.": ep_item.get("Notification No.", ""),
                "Notification Date": ep_item.get("Notification Date", ""),
                "chapter_no": ep_item.get("chapter_no", "")
            })

    return merged_data

if __name__ == "__main__":
    # Load RoDTEP and Export Policy JSONs
    with open("/content/sorted_hscode.json", "r") as f:
        rodtep_json = json.load(f)

    with open("/content/export_policy_chapter_hs.json", "r") as f:
        export_policy_json = json.load(f)

    # Merge only common codes
    merged_result = merge_common_trade_jsons(rodtep_json, export_policy_json)

    # Write to output file
    with open("merged_export_output.json", "w") as f:
        json.dump(merged_result, f, indent=2)

    print("✅ Merged (common only) output written to 'merged_export_output.json'")


✅ Merged (common only) output written to 'merged_export_output.json'


In [13]:
import json

def update_chapter_number(data):
    for item in data:
        hs_code = item.get("Tariff Lines / HS Code", "")
        if len(hs_code) >= 2 and hs_code[1].isdigit():
            # Use the second digit as chapter number
            item["chapter_no"] = hs_code[1]
        else:
            item["chapter_no"] = ""  # fallback if code is invalid
    return data

if __name__ == "__main__":
    # Load the merged JSON
    with open("/content/merged_export_output.json", "r") as f:
        merged_data = json.load(f)

    # Update chapter_no
    updated_data = update_chapter_number(merged_data)

    # Write the result to a new file
    with open("merged_with_chapter_updated.json", "w") as f:
        json.dump(updated_data, f, indent=2)

    print("✅ Chapter numbers updated and saved to 'merged_with_chapter_updated.json'")


✅ Chapter numbers updated and saved to 'merged_with_chapter_updated.json'


In [14]:
import json
import csv

# === Input and Output File Paths ===
json_file_path = '/content/merged_with_chapter_updated.json'   # Replace with your JSON file path
csv_file_path = 'output.csv'    # Replace with your desired CSV file path

# === Load JSON Data ===
with open(json_file_path, 'r', encoding='utf-8') as json_file:
    data = json.load(json_file)

# === Ensure JSON is a List of Dicts ===
if not isinstance(data, list):
    raise ValueError("JSON data should be a list of dictionaries.")

# === Write CSV ===
with open(csv_file_path, 'w', newline='', encoding='utf-8') as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=data[0].keys())
    writer.writeheader()
    writer.writerows(data)

print(f"✅ CSV written successfully to {csv_file_path}")


✅ CSV written successfully to output.csv
